# BERT Masked Language Modeling with Hugging Face Transformers

이 노트북은 Kaggle Note 환경에서 `transformers`의 `bert-base-uncased` 모델을 사용해 Masked Language Modeling을 수행하는 예제입니다.

구성:
- `bert-base-uncased` 모델 사용
- `AutoTokenizer`와 `AutoModelForMaskedLM` 사용
- `[MASK]` 토큰이 포함된 문장 입력
- PyTorch 기반 추론
- `[MASK]` 위치의 top-5 예측 단어 출력

In [ ]:
# Kaggle Note에서 필요한 라이브러리를 설치합니다.
# torch는 보통 기본 제공되지만, transformers는 없는 경우가 있어 함께 설치합니다.
%pip install -q transformers

In [ ]:
# 필요한 라이브러리를 불러옵니다.
import torch
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
# 사용할 디바이스를 설정합니다.
# Kaggle GPU가 활성화되어 있으면 CUDA를 사용하고, 아니면 CPU로 실행합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 토크나이저와 Masked Language Modeling용 BERT 모델을 불러옵니다.
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
model.to(device)
model.eval()

In [ ]:
# [MASK] 토큰이 포함된 입력 문장을 준비합니다.
# BERT는 문장 안의 [MASK] 위치에 어떤 단어가 올지 예측할 수 있습니다.
text = "The capital of France is [MASK]."
print(f"Input sentence: {text}")

In [ ]:
# 입력 문장을 토큰화하여 PyTorch 텐서로 변환합니다.
inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# [MASK] 토큰의 위치를 찾습니다.
mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

print(f"Mask token: {tokenizer.mask_token}")
print(f"Mask position: {mask_token_index.item()}")

In [ ]:
# PyTorch 기반으로 추론을 수행합니다.
# gradient 계산이 필요 없으므로 torch.no_grad()를 사용합니다.
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# [MASK] 위치의 로짓만 가져와 확률로 변환합니다.
mask_token_logits = logits[0, mask_token_index, :]
mask_token_probs = torch.softmax(mask_token_logits, dim=-1)

# top-5 후보 토큰을 구합니다.
top_k = 5
top_probs, top_indices = torch.topk(mask_token_probs, k=top_k, dim=-1)

In [ ]:
# top-5 예측 단어를 출력합니다.
print("Top-5 predictions for [MASK]:")

for rank, (token_id, score) in enumerate(zip(top_indices[0], top_probs[0]), start=1):
    predicted_token = tokenizer.decode([token_id]).strip()
    print(f"{rank}. {predicted_token:<15} {score.item() * 100:.2f}%")

In [ ]:
# 예측된 단어를 원래 문장에 넣어 예시 문장을 함께 출력합니다.
print("\nCompleted sentence examples:")

for rank, token_id in enumerate(top_indices[0], start=1):
    predicted_token = tokenizer.decode([token_id]).strip()
    completed_text = text.replace(tokenizer.mask_token, predicted_token)
    print(f"{rank}. {completed_text}")